# Build a background

byob builds a static plasma background for particle tracking in four steps: probe values
are spread over a grid (`fill_probes`), the bulk potential follows from Boltzmann's relation,
the sheath and presheath at each wall are added analytically (`sheath_drop`, `wall_layer`), and
the result is sampled as an `xarray.Dataset` or handed to plasmacoll as a `FieldProfile`.
This tutorial goes through each step on a small 2D chamber.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plasma_plots  # noqa: F401  (registers the .plasma accessor)
import xarray as xr
from plasmacoll.constants import PROTON_MASS
from simgeometry import Rectangle

from byob import (
    Background,
    FieldProfile,
    Probe,
    UniformGrid,
    Wall,
    debye_length,
    fill_probes,
    sheath_drop,
    sheath_width,
)
from byob.sheath import PlasmaState, wall_layer

## Probes on a grid

A `Probe` fixes one quantity at one position. `fill_probes` interpolates the probes of one
quantity over the nodes of a `UniformGrid`, in log space so that densities stay positive.
The default `"biharmonic"` method is smooth through the probes; `"harmonic"` never overshoots
but has a cusp at every probe.

In [ ]:
grid = UniformGrid((0.0, 0.0), (0.05, 0.02), (51, 21))
density_probes = [
    Probe("density", (0.01, 0.01), 2e17),
    Probe("density", (0.04, 0.005), 5e16),
    Probe("density", (0.03, 0.017), 1e17),
]
fig, axes = plt.subplots(1, 2, figsize=(10, 3), layout="constrained")
for ax, method in zip(axes, ["biharmonic", "harmonic"], strict=True):
    values = fill_probes(grid, density_probes, method=method)
    density = xr.DataArray(
        values,
        coords=dict(zip("xy", grid.axes, strict=True)),
        name="density",
        attrs={"units": "m^-3"},
    )
    density.plasma.plot.slice(x="x", y="y", ax=ax)
    ax.plot(*np.array([p.position for p in density_probes]).T, "wx")
    ax.set_title(method)
plt.show()

## Sheaths

A `Wall` is any simgeometry shape with a signed distance, floating or biased. The sheath is
not resolved on the grid: its drop and width follow from the local plasma state. For a
floating wall the drop is $\frac{T_e}{2}\ln\frac{m_i}{2\pi m_e(1+T_i/T_e)}$, below a
presheath drop of $T_e/2$; a biased wall below the sheath edge takes the rest down to its bias.

In [ ]:
bulk = PlasmaState(
    density=np.array([1e17]),
    electron_temperature=np.array([2.0]),
    ion_temperature=np.array([0.5]),
    potential=np.array([0.0]),
)
wall = Wall(Rectangle((-0.01, 0.01), (0.02, 0.04)))  # floating, solid for x < 0
biased = Wall(wall.shape, kind="biased", bias=-20.0)
for w in (wall, biased):
    drop = sheath_drop(w, bulk, PROTON_MASS)
    width = sheath_width(drop, 1e17 * np.exp(-0.5), 2.0)
    print(f"{w.kind:8s}: drop {drop[0]:5.2f} V, width {width[0] * 1e6:5.0f} um")
print(f"Debye length: {debye_length(1e17, 2.0) * 1e6:.0f} um")

`wall_layer` gives the densities and potential across the presheath and sheath at a
distance from the wall. Electrons follow Boltzmann's relation, ions conserve the Bohm flux,
so the ions outnumber the electrons in the sheath.

In [ ]:
distance = np.linspace(0, 1e-3, 400)
state = PlasmaState(*(np.full(distance.size, v[0]) for v in vars(bulk).values()))
presheath = 20 * debye_length(1e17, 2.0)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
for w in (wall, biased):
    layer = xr.Dataset(
        {
            q: ("distance", v)
            for q, v in wall_layer(distance, w, state, PROTON_MASS, presheath).items()
        },
        coords={"distance": distance},
    )
    units = {
        "distance": "m",
        "potential": "V",
        "ion_density": "m^-3",
        "electron_density": "m^-3",
    }
    for name, unit in units.items():
        layer[name].attrs["units"] = unit
    layer.potential.plasma.plot.lineout(x="distance", ax=axes[0])
    layer.ion_density.plasma.plot.lineout(x="distance", ax=axes[1])
    layer.electron_density.plasma.plot.lineout(x="distance", ax=axes[1])
axes[0].legend(["floating", "biased"], frameon=False)
axes[1].legend(
    ["ions, floating", "electrons, floating", "ions, biased", "electrons, biased"],
    frameon=False,
)
axes[1].set_title("densities")
plt.show()

## The background

`Background.build` does all of this together: it fills the probes around the walls (wall
nodes are left out, so values flow around them), sets the bulk potential, and adds the
controlling wall's layer wherever it is evaluated. Here a floating wall at $x=0$ and a
biased electrode at the far end bound a plasma with a density and temperature gradient.

In [ ]:
electrode = Wall(Rectangle((0.05, 0.01), (0.01, 0.04)), kind="biased", bias=-10.0)
probes = [
    *density_probes,
    Probe("electron_temperature", (0.01, 0.01), 3.0),
    Probe("electron_temperature", (0.04, 0.01), 1.5),
    Probe("ion_temperature", (0.025, 0.01), 0.5),
]
background = Background.build(grid, probes, [wall, electrode])

## Sampling

`sample` evaluates everything on a tensor grid and returns an `xarray.Dataset`; a scalar fixes
an axis. Points inside walls are NaN. The datasets plot directly with plasma-plots.

In [ ]:
maps = background.sample(resolution=120)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
maps.electron_temperature.plasma.plot.slice(x="x", y="y", ax=axes[0])
maps.potential.plasma.plot.slice(x="x", y="y", ax=axes[1])
plt.show()

In [ ]:
line = background.sample(y=0.01, resolution=600)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
line.potential.plasma.plot.lineout(x="x", ax=axes[0])
line.electron_density.plasma.plot.lineout(x="x", ax=axes[1])
line.ion_density.plasma.plot.lineout(x="x", ax=axes[1])
axes[1].legend(["electrons", "ions"], frameon=False)
plt.show()

## For particle tracking

`evaluate` and `electric_field` take `(N, ndim)` positions, as a particle pusher holds them.

In [ ]:
rng = np.random.default_rng(0)
positions = rng.uniform((0.0, 0.0), (0.045, 0.02), size=(5, 2))
state = background.evaluate(positions)
print(state["potential"])
print(background.electric_field(positions))

`profile` returns a `FieldProfile`: a quantity over a reference value, which plasmacoll
takes as a `Profile` to shape a collision background. The biharmonic fill overshoots the
largest density probe a little, so the reference is slightly above 2e17.

In [ ]:
profile = background.profile("electron_density")
assert isinstance(profile, FieldProfile)
print(f"reference {profile.reference:.2e} m^-3, max factor {profile.max_factor:.2f}")
print(profile(positions))